In [3]:
import os
from pathlib import Path
ROOT = "../" # Base directory relative to this notebook's location. Adjust if the notebook is moved
CSV_DIR = os.path.join(ROOT, "csv", "historical-data")
CSV_2015_2020 = os.path.join(CSV_DIR, "2015-2020")
CSV_2021_2025 = os.path.join(CSV_DIR, "2021-2025")
CSV_MERGE = os.path.join(CSV_DIR, "2015-2025")
import json
JSON_DIR = os.path.join(ROOT, "json")

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

import geopandas as gpd
MAPS_DIR = os.path.join(ROOT, "maps")
import contextily as ctx

from IPython.display import clear_output
import time

import sys
sys.path.append(ROOT)
import src.weather_stats as ws
from src.weather_data_download import download_station_data, cast_columns

In [4]:
source_dirs = [Path(CSV_2015_2020), Path(CSV_2021_2025)]
merge_dir = Path(CSV_MERGE)
merge_dir.mkdir(parents=True, exist_ok=True)

station_files = {}
for period_dir in source_dirs:
    for csv_file in period_dir.glob("*_hist.csv"):
        station_code = csv_file.name[:-9]
        station_files.setdefault(station_code, {})[period_dir.name] = csv_file

for station_code in sorted(station_files):
    print(f"Merging station {station_code}")

    frames = []
    for period_name in ["2015-2020", "2021-2025"]:
        csv_file = station_files[station_code].get(period_name)
        if csv_file is None:
            continue

        frame = pd.read_csv(csv_file)
        if "fecha" not in frame.columns:
            raise ValueError(f"Missing 'fecha' column in {csv_file}")

        frame["fecha"] = pd.to_datetime(frame["fecha"], format="%Y-%m-%d", errors="raise")
        frames.append(frame)

    merged = pd.concat(frames, ignore_index=True)
    merged = merged.sort_values("fecha", ascending=True, kind="mergesort").reset_index(drop=True)

    duplicate_dates = merged.loc[merged["fecha"].duplicated(keep=False), "fecha"]
    if not duplicate_dates.empty:
        duplicated_days = duplicate_dates.dt.strftime("%Y-%m-%d").drop_duplicates().tolist()
        raise ValueError(f"Duplicate fecha values found for {station_code}: {duplicated_days}")

    merged["fecha"] = merged["fecha"].dt.strftime("%Y-%m-%d")
    merged.to_csv(merge_dir / f"{station_code}_hist.csv", index=False)


Merging station 0009X
Merging station 0016A
Merging station 0034X
Merging station 0042Y
Merging station 0061X
Merging station 0066X
Merging station 0073X
Merging station 0076
Merging station 0092X
Merging station 0106X
Merging station 0114X
Merging station 0120X
Merging station 0149X
Merging station 0158X
Merging station 0171X
Merging station 0194D
Merging station 0201X
Merging station 0222X
Merging station 0244X
Merging station 0260X
Merging station 0281Y
Merging station 0284X
Merging station 0312X
Merging station 0320I
Merging station 0360X
Merging station 0363X
Merging station 0367
Merging station 0370E
Merging station 0385X
Merging station 0394X
Merging station 0411X
Merging station 0413A
Merging station 0421X
Merging station 0429X
Merging station 0433D
Merging station 1002Y
Merging station 1010X
Merging station 1012P
Merging station 1014A
Merging station 1021X
Merging station 1025A
Merging station 1025X
Merging station 1026X
Merging station 1033X
Merging station 1037X
Merging stat